# Baseline de scoring — paliers M0 et M1

**Objectif de ce notebook.** Construire et comparer les deux premiers paliers du modèle de scoring crédit :

- **M0** — variables *déclaratives* (celles du formulaire de demande) : âge, revenu déclaré, ancienneté, zone, secteur, région.
- **M1** — M0 **+** variables *comportementales* déduites des transactions (montants, flux, régularité), **sans jamais lire le sens** des libellés — cette lecture du texte est réservée aux paliers M2/M3.

Pour chaque palier, on ne se contente pas d'un seul entraînement : on **cherche le meilleur réglage** (régularisation `C` de la régression logistique) par validation croisée, puis on regarde **comment chaque variable pèse** dans le modèle final (table de coefficients), avant de comparer statistiquement M0 à M1.

**Règles respectées à chaque étape :**
- les colonnes `gt_*` (vérité-terrain) ne sont **jamais** utilisées comme variables — elles ne seraient pas connues en production ;
- `gt_categorie` n'est pas utilisée : comprendre le *sens* des libellés est réservé à M2+ (traitement du texte) ;
- `sexe` (attribut protégé) est exclu des variables du modèle, mais gardé de côté pour l'audit d'équité ;
- le prétraitement (mise à l'échelle, recherche de `C`) est ajusté sur le **train uniquement** — aucune fuite vers le test ;
- le déséquilibre de classes est traité par pondération des classes (`class_weight="balanced"`).

Le code générique (pipeline, recherche de `C`, coefficients, métriques, test de DeLong) vit dans [`scoring_utils.py`](scoring_utils.py) pour être réutilisé tel quel par les paliers suivants (M2, M3, ...).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from scoring_utils import (
    audit_equite,
    construire_pipeline,
    construire_variables_comportementales,
    delong_test,
    evaluer,
    extraire_coefficients,
    rechercher_meilleur_C,
)

SEED = 42
TARGET = "defaut_90j"
PROTEGE = ["sexe"]  # exclu des features, conservé pour l'audit d'équité

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]

# Grille de régularisation testée pour chaque palier : plus C est petit, plus
# le modèle est contraint (coefficients proches de 0) ; plus C est grand,
# plus il colle librement aux données d'entraînement.
GRILLE_C = [0.001, 0.01, 0.1, 1, 10, 100]

## Étape 1 — Charger et regarder les données

Deux fichiers : un client par ligne (`clients_synth.csv`) et une transaction par ligne (`transactions_synth.csv`). On regarde d'abord à quoi ils ressemblent, et le taux de défaut global, avant de construire quoi que ce soit.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

print(f"clients : {clients.shape[0]} lignes, {clients.shape[1]} colonnes")
print(f"transactions : {tx.shape[0]} lignes, {tx.shape[1]} colonnes")
print(f"taux de défaut global : {clients[TARGET].mean():.3f}\n")

clients.head()

clients : 2000 lignes, 17 colonnes
transactions : 91666 lignes, 6 colonnes
taux de défaut global : 0.159



,client_id,age,sexe,zone,secteur,region,revenu_declare,anciennete_mois,defaut_90j,gt_C_star,gt_income_stab,gt_distress,gt_hidden_support,gt_camf_intensity,gt_p_mensonge,gt_defaut_true,gt_revenu_reel
0,1,27,F,urbaine,informel,Nord,109000.0,38,0,-1.1363,-0.3835,0.8139,-0.6940,0.5174,0.4248,0,97000.0
1,2,26,M,urbaine,informel,Ouest,174000.0,71,0,0.2871,0.8949,-0.8814,-0.5429,0.7169,0.0545,0,174000.0
2,3,19,M,urbaine,formel,Sud,189000.0,9,0,-0.4672,0.8274,0.7714,-0.2002,0.5309,0.2418,0,170000.0
3,4,20,F,urbaine,informel,Extrême-Nord,33000.0,86,0,-0.3632,-1.6457,0.6543,0.8435,0.7543,0.2719,0,30000.0
4,5,28,F,semi_urbaine,formel,Ouest,162000.0,49,1,-1.3695,-0.0724,2.6820,-0.1867,0.6462,0.7158,1,116000.0


In [3]:
tx.head()

,client_id,date,montant,libelle,gt_categorie,gt_mensonge
0,1,2024-01-06,-6338.0,paiMt-FAct.EnEo/REF9731984,ELEC,0
1,1,2024-01-08,-1481199.0,paIEE LoYErBlR REF4806837 AGMAR 1/04,LOYER,0
2,1,2024-01-14,19238.0,eNVoIE Momo au boSs/TPE1761/25/08,MOMO,0
3,1,2024-01-14,-8337.0,OpR nJAN,NJANGUI,0
4,1,2024-01-15,-7971.0,DeBcotisAtION_TontInE REF383522 TPE50248,NJANGUI,0


## Les variables comportementales, une par une

On résume l'historique de transactions de chaque client en une seule ligne. Aucune de ces variables ne lit le *sens* des libellés — seulement des montants, des signes et des dates :

| Variable | Ce qu'elle capture |
|---|---|
| `nb_tx` | volume d'activité du compte (nombre de transactions) |
| `pct_debits` | part des transactions qui sont des débits (sorties d'argent) |
| `inflow` | total des entrées d'argent sur la période |
| `outflow` | total des sorties d'argent sur la période |
| `net_flow` | `inflow - outflow` : le compte s'enrichit-il ou se vide-t-il ? |
| `mean_abs` / `std_abs` / `max_abs` | montant moyen, dispersion et montant maximal des transactions |
| `cv_abs` | volatilité relative (`std_abs / mean_abs`) : compte stable ou erratique |
| `nb_jours_actifs` | nombre de jours distincts avec au moins une transaction |
| `tx_par_jour` | intensité d'usage du compte les jours actifs |
| `ecart_revenu` | écart relatif entre le revenu **déclaré** et le flux entrant **observé** (`inflow / 3` mois) — le signal le plus direct sur une déclaration optimiste ou honnête |

In [4]:
comp = construire_variables_comportementales(tx, clients)
comp.head()

,client_id,nb_tx,pct_debits,inflow,outflow,mean_abs,std_abs,max_abs,nb_jours_actifs,net_flow,cv_abs,tx_par_jour,ecart_revenu
0,1,28,0.821429,1970486.0,1958456.0,140319.357143,352120.284243,1481199.0,26,12030.0,2.509421,1.076923,-5.025951
1,2,58,0.810345,4128443.0,2782051.0,119146.448276,251905.005755,1009317.0,44,1346392.0,2.114247,1.318182,-6.908895
2,3,36,0.750000,2775876.0,2040831.0,133797.416667,292261.844375,1364223.0,25,735045.0,2.184361,1.440000,-3.895725
3,4,21,0.857143,179598.0,1871038.0,97649.333333,319716.352365,1489911.0,19,-1691440.0,3.274127,1.105263,-0.814121
4,5,29,0.931034,397596.0,4884317.0,182134.931034,357177.292957,1209253.0,20,-4486721.0,1.961059,1.450000,0.181901


## Étape 2 — Construire le jeu de données final et le split train/test

On fusionne les variables comportementales aux clients, on retire les colonnes `gt_*` (jamais utilisables comme variables) et `client_id`, puis on découpe en train (80 %) / test (20 %) de façon **stratifiée** (même taux de défaut des deux côtés). Ce même découpage servira à M0 **et** M1, pour que la comparaison entre les deux soit sur les mêmes clients de test.

In [5]:
df = clients.merge(comp, on="client_id", how="left")
df[COMPORTEMENTAL_NUM] = df[COMPORTEMENTAL_NUM].fillna(0)

y = df[TARGET].values
colonnes_gt = [c for c in df.columns if c.startswith("gt_")]
X = df.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte = df.loc[Xte.index]  # pour l'audit d'équité par secteur, plus loin

print(f"train {len(Xtr)} | test {len(Xte)}")
print(f"taux de défaut : train {ytr.mean():.3f} | test {yte.mean():.3f}")

train 1600 | test 400
taux de défaut : train 0.159 | test 0.158


## Étape 3 — Palier M0 : variables déclaratives

Ce sont les seules informations disponibles dans un formulaire de demande de crédit, avant toute analyse du compte :

| Variable | Pourquoi elle est là |
|---|---|
| `age` | proxy classique de stabilité financière et d'expérience de vie |
| `revenu_declare` | niveau de vie annoncé par le client lui-même |
| `anciennete_mois` | ancienneté de la relation bancaire — un client connu de longue date est mieux documenté |
| `zone` | urbain/rural — contexte économique local |
| `secteur` | formel/informel — stabilité de revenu très différente entre les deux |
| `region` | disparités économiques régionales |

`sexe` est volontairement absent : c'est un attribut protégé, gardé de côté pour l'audit d'équité, jamais utilisé comme variable prédictive.

**Recherche du meilleur `C`.** On ne fixe pas `C=1` par défaut : on teste plusieurs valeurs de régularisation par validation croisée sur le train, et on garde celle qui maximise l'AUC moyenne.

In [6]:
pipeline_m0 = construire_pipeline(DECLARATIF_NUM, DECLARATIF_CAT, seed=SEED)
modele_m0, cv_m0 = rechercher_meilleur_C(pipeline_m0, Xtr, ytr, GRILLE_C, seed=SEED)

print("AUC moyenne (validation croisée, train) selon C :")
cv_m0

AUC moyenne (validation croisée, train) selon C :


,C,auc_cv_moyen,auc_cv_ecart_type
0,0.001,0.677295,0.028864
1,0.010,0.673980,0.027130
2,0.100,0.666596,0.026184
3,1.000,0.662410,0.025905
4,10.000,0.661723,0.026045
5,100.000,0.661752,0.025961


In [7]:
meilleur_c_m0 = modele_m0.named_steps["clf"].C
print(f"meilleur C retenu pour M0 : {meilleur_c_m0}\n")

p0 = modele_m0.predict_proba(Xte)[:, 1]
resultat_m0 = evaluer("M0", yte, p0)

coefs_m0 = extraire_coefficients(modele_m0, DECLARATIF_NUM, DECLARATIF_CAT)
coefs_m0

meilleur C retenu pour M0 : 0.001

M0   | AUC 0.653 | Gini 0.307 | KS 0.268


,variable,coefficient
0,revenu_declare,-0.150441
1,secteur_formel,-0.067735
2,secteur_informel,0.067730
3,anciennete_mois,-0.020496
4,age,-0.007684
5,region_Adamaoua,0.006511
6,region_Centre,-0.006176
7,region_Extrême-Nord,-0.005894
8,region_Sud-Ouest,0.004007
9,region_Littoral,0.003891


**Comment lire la table de coefficients.** Les variables numériques sont standardisées et les catégorielles sont en one-hot : les coefficients sont donc comparables entre eux. Un coefficient **positif** augmente le risque de défaut prédit, un coefficient **négatif** le diminue ; plus la valeur absolue est grande, plus la variable pèse dans la décision de M0.

## Étape 4 — Palier M1 : + comportement du compte

Même variables déclaratives que M0, **plus** les 12 variables comportementales détaillées plus haut (`nb_tx`, `pct_debits`, `inflow`, ..., `ecart_revenu`). Même logique : on cherche le meilleur `C` sur le train avant d'évaluer sur le test.

In [8]:
num_m1 = DECLARATIF_NUM + COMPORTEMENTAL_NUM

pipeline_m1 = construire_pipeline(num_m1, DECLARATIF_CAT, seed=SEED)
modele_m1, cv_m1 = rechercher_meilleur_C(pipeline_m1, Xtr, ytr, GRILLE_C, seed=SEED)

print("AUC moyenne (validation croisée, train) selon C :")
cv_m1

AUC moyenne (validation croisée, train) selon C :


,C,auc_cv_moyen,auc_cv_ecart_type
0,0.001,0.675600,0.042405
1,0.010,0.684135,0.039116
2,0.100,0.683035,0.038964
3,1.000,0.681808,0.041637
4,10.000,0.682434,0.042368
5,100.000,0.682375,0.042466


In [9]:
meilleur_c_m1 = modele_m1.named_steps["clf"].C
print(f"meilleur C retenu pour M1 : {meilleur_c_m1}\n")

p1 = modele_m1.predict_proba(Xte)[:, 1]
resultat_m1 = evaluer("M1", yte, p1)

coefs_m1 = extraire_coefficients(modele_m1, num_m1, DECLARATIF_CAT)
coefs_m1

meilleur C retenu pour M1 : 0.01

M1   | AUC 0.666 | Gini 0.333 | KS 0.275


,variable,coefficient
0,secteur_formel,-0.295975
1,secteur_informel,0.295748
2,tx_par_jour,-0.155393
3,nb_jours_actifs,-0.138432
4,cv_abs,0.111687
5,nb_tx,-0.107329
6,inflow,-0.105541
7,outflow,-0.105508
8,ecart_revenu,0.070377
9,anciennete_mois,-0.066943


**Lecture.** Si les variables déclaratives (`age`, `revenu_declare`, ...) gardent des coefficients proches de ceux de M0, c'est signe que M1 ne fait qu'*ajouter* de l'information sans la contredire. Si une variable comportementale arrive en tête (souvent `ecart_revenu` ou `net_flow`), c'est elle qui explique le déplacement de l'AUC entre M0 et M1.

Mais avant de comparer sérieusement M0 et M1, un point mérite d'être vérifié : les deux tables de coefficients ci-dessus sont-elles vraiment comparables telles quelles ?

### Un piège : comparer des coefficients à `C` différent

M0 a retenu `C=0,001` (régularisation très forte) et M1 a retenu `C=0,01` (10× moins forte) — chaque recherche a choisi le `C` qui maximise l'AUC pour *son propre* palier, pas pour rendre les deux tables de coefficients comparables entre elles. Une partie de l'écart observé plus haut (`revenu_declare` qui perd du poids, `secteur` qui en gagne beaucoup) peut donc venir de ce simple changement d'échelle de régularisation, et pas d'un vrai effet des variables comportementales.

Pour isoler l'effet propre de la **colinéarité** (les variables comportementales qui recouvrent une partie du signal déjà présent dans les variables déclaratives — par exemple `ecart_revenu` est calculée à partir de `revenu_declare`), on refait tourner M0 et M1 à **`C` identique et fixe** (`C=1`, la valeur par défaut de scikit-learn, non optimisée). Les AUC obtenues ici ne sont **pas** les résultats officiels du notebook — ceux restent ceux de la recherche de `C` plus haut — cette comparaison ne sert qu'à lire les coefficients sur un pied d'égalité.

In [10]:
C_FIXE = 1.0  # valeur unique imposée aux deux modèles, uniquement pour cette comparaison

pipeline_m0_fixe = construire_pipeline(DECLARATIF_NUM, DECLARATIF_CAT, seed=SEED)
pipeline_m0_fixe.set_params(clf__C=C_FIXE).fit(Xtr, ytr)
coefs_m0_fixe = extraire_coefficients(pipeline_m0_fixe, DECLARATIF_NUM, DECLARATIF_CAT)

pipeline_m1_fixe = construire_pipeline(num_m1, DECLARATIF_CAT, seed=SEED)
pipeline_m1_fixe.set_params(clf__C=C_FIXE).fit(Xtr, ytr)
coefs_m1_fixe = extraire_coefficients(pipeline_m1_fixe, num_m1, DECLARATIF_CAT)

# seules les variables communes aux deux modèles (déclaratives) survivent à la fusion
comparaison = coefs_m0_fixe.merge(coefs_m1_fixe, on="variable", suffixes=("_M0", "_M1"))
comparaison["ecart"] = comparaison.coefficient_M1 - comparaison.coefficient_M0
comparaison = (comparaison
               .reindex(comparaison.ecart.abs().sort_values(ascending=False).index)
               .reset_index(drop=True))

print(f"C fixé à {C_FIXE} pour M0 et M1 (comparaison uniquement, pas le résultat officiel du notebook)\n")
comparaison

C fixé à 1.0 pour M0 et M1 (comparaison uniquement, pas le résultat officiel du notebook)



,variable,coefficient_M0,coefficient_M1,ecart
0,revenu_declare,-0.492748,0.556612,1.049360
1,secteur_formel,-0.398198,-0.714175,-0.315976
2,secteur_informel,0.315517,0.614835,0.299318
3,region_Nord-Ouest,0.080986,0.013406,-0.067581
4,region_Centre,-0.154375,-0.204526,-0.050151
5,region_Est,-0.074751,-0.027300,0.047451
6,region_Sud,0.018343,-0.026010,-0.044353
7,region_Ouest,-0.044051,-0.003949,0.040102
8,region_Adamaoua,0.161250,0.194747,0.033497
9,zone_semi_urbaine,-0.021136,-0.051802,-0.030666


**Lecture.** `C` étant maintenant identique des deux côtés, tout écart restant entre `coefficient_M0` et `coefficient_M1` n'est plus un artefact de réglage : il vient bien de la colinéarité avec les variables comportementales ajoutées dans M1.

Le cas le plus frappant est `revenu_declare`, qui ne se contente pas de perdre du poids : son coefficient **change de signe** entre M0 et M1 (négatif dans M0, positif dans M1). Seul, un revenu déclaré élevé est associé à *moins* de risque (lecture naturelle : plus de revenu, plus de capacité de remboursement). Mais une fois `ecart_revenu` dans le modèle — qui compare justement ce revenu déclaré au flux réellement observé — le modèle réinterprète `revenu_declare` : à comportement de compte identique, déclarer un revenu élevé devient plutôt un signal de risque, probablement parce que c'est `ecart_revenu` qui porte maintenant l'information utile (revenu déclaré crédible ou non), laissant à `revenu_declare` seul un rôle résiduel inversé. C'est un exemple classique de suppression variable en régression : une variable corrélée à une autre déjà présente peut voir son coefficient se retourner sans que cela remette en cause le modèle — mais cela interdit de lire `revenu_declare` isolément dans M1.

`secteur`, à l'inverse, garde le même signe mais un poids plus fort dans M1 (`coefficient_M0` ≈ -0,40, `coefficient_M1` ≈ -0,71 pour `secteur_formel`) : une fois qu'on contrôle pour l'usage réel du compte, l'écart de risque propre au secteur ressort encore plus nettement — pas un artefact, un vrai effet.

## Étape 5 — M0 vs M1 : le gain est-il réel ?

M0 et M1 (les modèles à `C` optimisé par validation croisée, pas ceux de la comparaison ci-dessus) sont évalués sur **les mêmes clients de test** : leurs AUC sont donc corrélées, et une simple comparaison de chiffres ne dit pas si l'écart est réel ou dû au hasard de l'échantillon. Le test de DeLong (Sun & Xu, 2014) répond à cette question précise.

In [11]:
aucs, pval = delong_test(yte, p0, p1)
conclusion = "gain significatif" if pval < 0.05 else "non significatif"

print(f"DeLong M0 vs M1 : AUC {aucs[0]:.3f} -> {aucs[1]:.3f} | p = {pval:.2e} -> {conclusion}")

DeLong M0 vs M1 : AUC 0.653 -> 0.666 | p = 4.76e-01 -> non significatif


## Étape 6 — Audit d'équité préliminaire

Aperçu rapide, au seuil d'approbation médian de M1 : le taux d'approbation est-il équilibré entre secteur formel et informel ? La **règle des 4/5** (courante en équité algorithmique) considère qu'un ratio d'approbation minoritaire/majoritaire **inférieur à 0,8** signale un *disparate impact* à traiter — c'est l'objet du palier équité, pas de ce notebook, mais le signal doit être visible dès maintenant.

In [12]:
taux_approbation, ratio, _ = audit_equite(dfte, p1, "secteur")

taux d'approbation par secteur : {'formel': 0.738, 'informel': 0.3}
ratio (règle des 4/5) : 0.41 -> disparate impact


## Lecture des résultats

**M0 → M1 : un comportement encore "muet".** Ajouter le comportement transactionnel brut (volumes, flux, régularité) sans jamais lire le sens des libellés donne un léger mieux en AUC, mais le test de DeLong ne le confirme généralement pas comme statistiquement significatif à ce stade. Les chiffres seuls (montants, signes, dates) ne suffisent pas encore à distinguer nettement les profils de risque : le signal utile est ailleurs — probablement dans la *catégorie* de chaque transaction, que M2 ira chercher en interprétant les libellés.

**Équité — à ne pas perdre de vue.** L'écart de taux d'approbation entre secteur formel et informel, déjà visible à ce stade avec les seules variables déclaratives et comportementales, est le signal que le palier équité devra mesurer précisément et corriger — voir [`m2_m3_texte.ipynb`](m2_m3_texte.ipynb) pour la suite (catégorisation des libellés, représentation vectorielle du texte, et premier audit détaillé du refus des "vrais bons" par groupe).